# CAFA 6 - Model Training

Train classifiers on precomputed ESM-2 embeddings:
1. LightGBM (one binary classifier per GO term)
2. MLP (multi-output neural network)

**No GPU needed** - runs on CPU with precomputed embeddings.

In [ ]:
# === Google Colab Setup (skip if running locally) ===
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    !git clone https://github.com/AyushSharma173/CafaChallenge.git
    %cd CafaChallenge
    !pip install -q obonet biopython h5py lightgbm scikit-learn matplotlib seaborn pyyaml
    
    # Kaggle API auth - upload your kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True)
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        from google.colab import files
        print('Upload your kaggle.json (get it from https://kaggle.com/settings > API > Create New Token)')
        uploaded = files.upload()
        !mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
    
    !kaggle competitions download -c cafa-6-protein-function-prediction -p data/raw
    !unzip -qo data/raw/cafa-6-protein-function-prediction.zip -d data/raw/
    %cd notebooks
    print('Colab setup complete!')

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from cafa6.config import Config
from cafa6.data_loader import (
    load_train_terms, load_taxonomy,
    build_label_matrix, create_cv_split
)
from cafa6.go_utils import load_go_graph, ASPECT_FULL_NAME
from cafa6.embeddings import load_embeddings
from cafa6.metrics import compute_fmax
from cafa6.models import NaiveFrequency, LightGBMMultilabel, MLPMultilabel

cfg = Config.from_yaml('../configs/default.yaml')
DATA_DIR = Path('../data/raw')

## 1. Load Data

In [ ]:
go_graph = load_go_graph(DATA_DIR / 'Train' / 'go-basic.obo')
terms_df = load_train_terms(DATA_DIR / 'Train' / 'train_terms.tsv')
taxonomy_df = load_taxonomy(DATA_DIR / 'Train' / 'train_taxonomy.tsv')

# Load precomputed embeddings
emb_file = Path(cfg.embeddings_dir) / f'train_{cfg.embeddings.model_name}.h5'
embeddings, emb_ids = load_embeddings(emb_file)
emb_lookup = {pid: emb for pid, emb in zip(emb_ids, embeddings)}
print(f'Embeddings: {embeddings.shape} ({embeddings.dtype})')

## 2. Prepare Data Per Ontology

In [ ]:
data = {}  # {aspect: (X_train, Y_train, X_val, Y_val, term_ids)}

for aspect in cfg.ontologies:
    label_matrix, pids, tids = build_label_matrix(
        terms_df, go_graph, aspect, min_count=cfg.min_term_count
    )
    
    # Filter to proteins with embeddings
    has_emb = [i for i, p in enumerate(pids) if p in emb_lookup]
    pids_f = [pids[i] for i in has_emb]
    labels_f = label_matrix[has_emb]
    X = np.array([emb_lookup[p] for p in pids_f], dtype=np.float32)
    
    # Split
    train_ids, val_ids = create_cv_split(pids_f, taxonomy_df, cfg.val_fraction, cfg.seed)
    train_set = set(train_ids)
    train_mask = np.array([p in train_set for p in pids_f])
    
    data[aspect] = {
        'X_train': X[train_mask], 'Y_train': labels_f[train_mask],
        'X_val': X[~train_mask], 'Y_val': labels_f[~train_mask],
        'term_ids': tids
    }
    print(f'{ASPECT_FULL_NAME[aspect]}: train={data[aspect]["X_train"].shape}, '
          f'val={data[aspect]["X_val"].shape}, terms={len(tids)}')

## 3. LightGBM Training

In [ ]:
lgb_results = {}

for aspect in cfg.ontologies:
    d = data[aspect]
    print(f'\n=== {ASPECT_FULL_NAME[aspect]} ===')
    
    model = LightGBMMultilabel(
        n_estimators=cfg.lightgbm.n_estimators,
        learning_rate=cfg.lightgbm.learning_rate,
        num_leaves=cfg.lightgbm.num_leaves,
    )
    model.fit(d['X_train'], d['Y_train'])
    
    val_scores = model.predict(d['X_val'])
    fmax, threshold = compute_fmax(d['Y_val'], val_scores)
    lgb_results[aspect] = {'fmax': fmax, 'threshold': threshold}
    print(f'{ASPECT_FULL_NAME[aspect]} LightGBM Fmax: {fmax:.4f} (t={threshold:.2f})')
    
    # Save model
    model.save(f'../models/lightgbm_{aspect}.pkl')
    np.save(f'../models/term_ids_{aspect}.npy', d['term_ids'])

avg = np.mean([r['fmax'] for r in lgb_results.values()])
print(f'\nLightGBM Average Fmax: {avg:.4f}')

## 4. MLP Training

In [ ]:
mlp_results = {}

for aspect in cfg.ontologies:
    d = data[aspect]
    print(f'\n=== {ASPECT_FULL_NAME[aspect]} ===')
    
    model = MLPMultilabel(
        input_dim=d['X_train'].shape[1],
        hidden_dims=cfg.mlp.hidden_dims,
        dropout=cfg.mlp.dropout,
        lr=cfg.mlp.lr,
        epochs=cfg.mlp.epochs,
        batch_size=cfg.mlp.batch_size,
    )
    losses = model.fit(d['X_train'], d['Y_train'], d['X_val'], d['Y_val'])
    
    val_scores = model.predict(d['X_val'])
    fmax, threshold = compute_fmax(d['Y_val'], val_scores)
    mlp_results[aspect] = {'fmax': fmax, 'threshold': threshold, 'losses': losses}
    print(f'{ASPECT_FULL_NAME[aspect]} MLP Fmax: {fmax:.4f} (t={threshold:.2f})')
    
    model.save(f'../models/mlp_{aspect}.pkl')

avg = np.mean([r['fmax'] for r in mlp_results.values()])
print(f'\nMLP Average Fmax: {avg:.4f}')

## 5. Compare Models

In [ ]:
# Also run frequency baseline for comparison
freq_results = {}
for aspect in cfg.ontologies:
    d = data[aspect]
    model = NaiveFrequency()
    model.fit(d['X_train'], d['Y_train'])
    val_scores = model.predict(d['X_val'])
    fmax, t = compute_fmax(d['Y_val'], val_scores)
    freq_results[aspect] = {'fmax': fmax}

# Comparison table
comparison = pd.DataFrame({
    'Frequency': {ASPECT_FULL_NAME[a]: freq_results[a]['fmax'] for a in cfg.ontologies},
    'LightGBM': {ASPECT_FULL_NAME[a]: lgb_results[a]['fmax'] for a in cfg.ontologies},
    'MLP': {ASPECT_FULL_NAME[a]: mlp_results[a]['fmax'] for a in cfg.ontologies},
})
comparison.loc['Average'] = comparison.mean()
print(comparison.round(4).to_string())

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
comparison.iloc[:-1].plot(kind='bar', ax=ax)
ax.set_ylabel('Fmax')
ax.set_title('Model Comparison by Ontology')
ax.legend(loc='lower right')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Training curves for MLP
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, aspect in enumerate(cfg.ontologies):
    axes[idx].plot(mlp_results[aspect]['losses'])
    axes[idx].set_xlabel('Epoch')
    axes[idx].set_ylabel('Val Loss')
    axes[idx].set_title(f'{ASPECT_FULL_NAME[aspect]} MLP Training')
plt.tight_layout()
plt.show()